### Validation Results: Single Report Extraction (Jev)

- **Macro-averaged ROC AUC:** `0.8544`
- **Evaluation:** Evaluated across the 12 knee abnormality targets using continuous probability predictions against ground-truth labels.

In [ ]:
!pip install typesafe_sdk

In [ ]:
import os, random, json
from itertools import combinations
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from typesafe_sdk import Choice, TypeSafeClient

os.environ["TYPESAFE_API_KEY"] = "YOUR_API_KEY" # fill this with your TypeSafe  API key 

DISEASES = {
    "ACL": "anterior cruciate ligament injury",
    "MCL": "medial collateral ligament injury",
    "Medial Meniscus": "medial meniscus tear",
    "Lateral Meniscus": "lateral meniscus tear",
    "Medial OA": "osteoarthritis of the medial tibiofemoral compartment",
    "Lateral OA": "osteoarthritis of the lateral tibiofemoral compartment",
    "PF OA": "patellofemoral osteoarthritis",
    "Effusion": "joint effusion / excess fluid",
    "Synovitis": "inflammation of the joint lining",
    "Baker's": "Baker's cyst",
    "Contusion": "bone contusion / bone bruise",
    "Fracture": "fracture"
}

In [ ]:
df_train = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")
df_train_known = df_train.loc[df_train.notna().all(axis=1)].reset_index(drop=True)
print(f"df_train: {len(df_train)}, df_train.unique: {df_train['Report'].nunique()}, df_train_known: {len(df_train_known)}")

In [ ]:
SHARED_CRITERIA = {
    "0": "The target structure is explicitly described as normal, intact, preserved, or without pathology (explicitly negated).",
    "1": "The target abnormality is explicitly reported as present, torn, degenerated, or probable.",
    "9": "The target structure is not mentioned, silent, or there is insufficient information in the text.",
}

TARGET_INSTRUCTIONS = {
    "ACL": (
        "Classify Anterior Cruciate Ligament (ACL) pathology. "
        "Select '1' for explicit tear, rupture, discontinuity, abnormal signal, injury, or ruptured post-surgical graft. "
        "An intact graft, intact native ligament, normal, preserved, or without pathology is '0'. If not mentioned, select '9'."
    ),
    "MCL": (
        "Classify Medial Collateral Ligament (MCL) pathology. "
        "Select '1' for explicit tear, rupture, sprain, abnormal signal, thickening, or injury. "
        "Select '0' if explicitly normal, intact, or preserved. If not mentioned, select '9'."
    ),
    "Medial Meniscus": (
        "Classify Medial Meniscus pathology. "
        "Select '1' for explicit tear, rupture, truncation, extrusion, displaced fragment, maceration, or abnormal morphology. "
        "Select '0' if explicitly normal or intact. If not mentioned, select '9'."
    ),
    "Lateral Meniscus": (
        "Classify Lateral Meniscus pathology. "
        "Select '1' for explicit tear, rupture, truncation, extrusion, displaced fragment, maceration, or abnormal morphology. "
        "Select '0' if explicitly normal or intact. If not mentioned, select '9'."
    ),
    "Medial OA": (
        "Classify Osteoarthritis of the MEDIAL tibiofemoral compartment. "
        "Select '1' for medial cartilage loss, chondrosis, thinning, joint space narrowing, marginal osteophytes, "
        "or subchondral sclerosis/cysts in the medial compartment. Select '0' if explicitly normal. If not mentioned, select '9'."
    ),
    "Lateral OA": (
        "Classify Osteoarthritis of the LATERAL tibiofemoral compartment. "
        "Select '1' for lateral cartilage loss, chondrosis, thinning, joint space narrowing, marginal osteophytes, "
        "or subchondral sclerosis/cysts in the lateral compartment. Select '0' if explicitly normal. If not mentioned, select '9'."
    ),
    "PF OA": (
        "Classify Patellofemoral Osteoarthritis (PF OA). "
        "Select '1' for patellofemoral cartilage loss, chondrosis, patellar/trochlear osteophytes, or subchondral changes. "
        "Select '0' if explicitly normal. If not mentioned, select '9'."
    ),
    "Effusion": (
        "Classify Joint Effusion (abnormally increased intra-articular fluid). "
        "Select '1' for moderate, large, marked, or clinically significant joint effusion, fluid distending suprapatellar recess/bursa, "
        "or effusion emphasized in conclusion/impression. "
        "Select '0' if explicitly absent ('no effusion') OR if fluid is only described as minimal, trace, small, or physiological incidental fluid. "
        "Fluid confined solely to a Baker cyst or soft tissues is NOT a joint effusion. If not mentioned, select '9'."
    ),
    "Synovitis": (
        "Classify Synovitis / Synovial Inflammation. "
        "Select '1' for explicit synovitis, synovial thickening, synovial hypertrophy/proliferation, villonodular changes, "
        "Hoffa fat pad impingement/edema/signal change, symptomatic plica thickening, or inflammatory suprapatellar bursitis. "
        "Note: Joint fluid alone without synovial thickening/Hoffa edema is NOT synovitis. "
        "Select '0' if explicitly normal/absent. If not mentioned, select '9'."
    ),
    "Baker's": (
        "Classify Baker's Cyst (popliteal cyst / gastrocnemius-semimembranosus bursa fluid). "
        "Select '1' for explicit Baker's cyst or popliteal cyst. "
        "Select '0' if explicitly absent or if only physiological trace fluid in normal bursa is described. If not mentioned, select '9'."
    ),
    "Contusion": (
        "Classify Bone Contusion / Bone Bruise (trabecular microfracture with marrow edema). "
        "Select '1' for explicit bone contusion, bone bruise, or traumatic marrow edema. "
        "Do NOT select 1 for isolated subchondral edema purely attributed to chronic osteoarthritis without trauma wording. "
        "Select '0' if explicitly absent. If not mentioned, select '9'."
    ),
    "Fracture": (
        "Classify Fracture of any bone about the knee. "
        "Select '1' for explicit acute/subacute fracture, cortical disruption, avulsion fracture, occult fracture, or insufficiency fracture. "
        "Select '0' if explicitly absent. If not mentioned, select '9'."
    ),
}

def build_jev_questions() -> dict[str, Choice]:
    questions = {}
    for target, instruction in TARGET_INSTRUCTIONS.items():
        questions[target] = Choice(instructions=f"{instruction}", criteria=SHARED_CRITERIA)
    return questions

def format_report_state(report_text: str) -> str:
    return (
        "Act as an expert musculoskeletal radiologist.\n"
        "Read the report in its original language (e.g., English, German, French, Spanish, etc.).\n"
        "Use ONLY explicitly stated findings; ignore clinical history, indications, or outside assumptions.\n"
        f"KNEE MRI RADIOLOGY REPORT:\n"
        f"\"\"\"\n{report_text.strip()}\n\"\"\"\n\n"
        f"Task: Extract the status of all requested knee structures based strictly on the text above."
    )

def classify_reports(reports: list[str], cache_path: str = "jev_labels.json"):

    client = TypeSafeClient()
    questions = build_jev_questions()

    results = {}
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            results = json.load(f)

    pending = [r for r in reports if r not in results]
    print(f"Total reports: {len(reports)} | Remaining to classify: {len(pending)}")

    for idx, report in enumerate(tqdm(pending, desc="Processing with Jev", ncols=100)):
        if not report or not report.strip():
            print(f"Skipping empty report: {idx}")
            continue
            
        try:
            state = format_report_state(report)
            response = client.system_one(state=state, questions=questions)
            
            report_summary = {}
            for target in TARGET_INSTRUCTIONS.keys():
                answer = response.answers[target]
                report_summary[target] = answer.probabilities
                
            results[report] = report_summary
            
            # Checkpoint to disk every 100 reports
            if (idx + 1) % 100 == 0:
                with open(cache_path, "w") as f:
                    json.dump(results, f)
                    
        except Exception as err:
            print(f"\nFailed on report: {report[:5]}... | Error: {err}")
            continue

    with open(cache_path, "w") as f:
        json.dump(results, f)

    return results

In [ ]:
list_reports = list(df_train['Report'].unique())
results = classify_reports(list_reports, cache_path="labels_jev_v2.json")

In [ ]:
results = json.load(open("Cache/single_jev_v2.json", "r"))
df_results = pd.DataFrame.from_dict(results, orient="index")
df_results.index.name = "Report"
df_report_flat = pd.DataFrame(index=df_results.index)
for disease in DISEASES:
    df_report_flat[f"{disease}_0"] = df_results[disease].map(lambda result: result.get("0") if isinstance(result, dict) else pd.NA)
    df_report_flat[f"{disease}_1"] = df_results[disease].map(lambda result: result.get("1") if isinstance(result, dict) else pd.NA)
    df_report_flat[f"{disease}_9"] = df_results[disease].map(lambda result: result.get("9") if isinstance(result, dict) else pd.NA)
df_report_flat.index.name = "Report"
df_label = df_train[['StudyInstanceUID', 'Report']].merge(df_report_flat, on="Report")
df_label.to_csv("Label_Extraction_Jev_v2.csv", index=False)

In [ ]:
df_label = pd.read_csv("/kaggle/input/datasets/hosseini7798/rsna-knee-label-extraction-jev/Label_Extraction_Jev_v2.csv")
df_label_known = df_label.loc[df_label['StudyInstanceUID'].isin(df_train_known['StudyInstanceUID'])]

In [ ]:
y_true = df_train_known[list(DISEASES)].values
y_pred = df_label_known[[f"{d}_1" for d in DISEASES.keys()]].values
roc_auc_score(y_true, y_pred, average='macro')